In [ ]:
import numpy as np
from keras.datasets import mnist

#trainign data
(x_train, y_train), (x_test, y_test) = mnist.load_data()

x_train = x_train / 255.0
x_test = x_test / 255.0

def one_hot(labels, num_classes=10):
    one_hot_labels = np.zeros((len(labels), num_classes))
    for i, label in enumerate(labels):
        one_hot_labels[i][label] = 1
    return one_hot_labels

y_train_oh = one_hot(y_train)
y_test_oh = one_hot(y_test)

In [ ]:
class CNN:
  """

  The goal of this project is to build a working CNN for classifying MNIST dataset using only python and numpy

  """
  def __init__ (self, input_array, kernal_size, pad_width=1, constant_value=0, pooling_size=2, layer_one_size=8, layer_two_size=16, learning_rate=0.01):
    #initializing constant variables
    self.pad_width = pad_width
    self.constant_value = constant_value
    self.input_array = np.pad(input_array, pad_width=self.pad_width, constant_values=self.constant_value)
    self.kernal_size = kernal_size
    self.pooling_size = pooling_size
    self.layer_one_size = layer_one_size
    self.layer_two_size = layer_two_size
    self.learning_rate = learning_rate

    #making randomized kernals
    self.kernals_layer_one = np.random.randn(self.layer_one_size, 1, self.kernal_size, self.kernal_size) * np.sqrt(2.0 / (kernal_size * kernal_size))
    self.kernals_layer_two = np.random.randn(self.layer_two_size, self.layer_one_size, self.kernal_size, self.kernal_size) * np.sqrt(2.0 / (layer_one_size * kernal_size * kernal_size))

    #making biases
    self.biases_layer_one = np.zeros(self.layer_one_size)
    self.biases_layer_two = np.zeros(self.layer_two_size)

    #fully connected layers
    self.flattened_size = None
    self.num_classes = 10
    self.fc_weights = None
    self.fc_biases = None

  """

  FORWARD PASS HELPER FUNCTIONS

  """

  #convoluting operation function
  def convolution(self, input_array, kernal):
      output = []
      for i in range(input_array.shape[0] - self.kernal_size + 1):
        row = []
        for j in range(input_array.shape[1] - self.kernal_size + 1):
          block = input_array[i:i+self.kernal_size, j:j+self.kernal_size]
          val = np.sum(block * kernal)
          row.append(val)
        output.append(row)
      return np.array(output)


  #RELU activation function
  def RELU(self, input_array):
      return np.maximum(0, input_array)

  #pooling function
  def max_pooling(self, input_array):
    rows, cols = input_array.shape
    out_rows = rows // self.pooling_size
    out_cols = cols // self.pooling_size
    output = np.zeros((out_rows, out_cols))
    mask = np.zeros_like(input_array)  # 1 where the max was taken
    for i in range(out_rows):
      for j in range(out_cols):
        block = input_array[i * self.pooling_size:(i + 1) * self.pooling_size, j * self.pooling_size:(j + 1) * self.pooling_size]
        max_val = np.max(block)
        output[i, j] = max_val
        # Mark the position of the maximum in the mask
        block_mask = (block == max_val)
        mask[i * self.pooling_size:(i + 1) * self.pooling_size,
          j * self.pooling_size:(j + 1) * self.pooling_size] = block_mask
    return output, mask

  #multi_channel_convolutions for layer two
  def multi_channel_convolution(self, input_maps, kernal):
    result = None
    for i in range(input_maps.shape[0]):
      conv = self.convolution(input_maps[i], kernal[i])
      if result is None:
        result = conv
      else:
        result += conv
    return result

  """

  Forward Pass Functions

  """

  #layer one
  def layer_one(self):
    feature_maps = []
    pre_relu_maps = []
    pool_masks = []
    for k in range(self.layer_one_size):
      kernal = self.kernals_layer_one[k][0]
      bias = self.biases_layer_one[k]

      #convolution
      conv = self.convolution(self.input_array, kernal)
      conv += bias
      pre_relu_maps.append(conv.copy())
      activated = self.RELU(conv)
      pooled, mask = self.max_pooling(activated)
      feature_maps.append(pooled)
      pool_masks.append(mask)


    return np.array(feature_maps), np.array(pre_relu_maps), np.array(pool_masks)

  #layer two
  def layer_two(self, input_maps):
    feature_maps = []
    pre_relu_maps = []
    pool_masks = []
    for k in range(self.layer_two_size):
      kernal = self.kernals_layer_two[k]
      bias = self.biases_layer_two[k]
      conv = self.multi_channel_convolution(input_maps, kernal)
      conv += bias
      pre_relu_maps.append(conv.copy())
      activated = self.RELU(conv)
      pooled, mask = self.max_pooling(activated)
      feature_maps.append(pooled)
      pool_masks.append(mask)

    return np.array(feature_maps), np.array(pre_relu_maps), np.array(pool_masks)

  #flattening
  def flatten(self, input_maps):
      return input_maps.flatten()

  """

  Backpropogation helper functions

  """
  #max pooling but backwards
  def pool_backward(self, d_pooled, mask):
    d_input = np.zeros_like(mask, dtype = float)
    out_rows, out_cols = d_pooled.shape
    for i in range(out_rows):
      for j in range(out_cols):
        d_input[i*self.pooling_size:(i+1)*self.pooling_size, j*self.pooling_size:(j+1)*self.pooling_size] += mask[i*self.pooling_size:(i+1)*self.pooling_size, j*self.pooling_size:(j+1)*self.pooling_size] * d_pooled[i,j]
    return d_input

  #RELU but backwards
  def RELU_backward(self, d_out, pre_relu):
    return d_out*(pre_relu>0)

  #Convoluting backwards
  def conv_backwards(self, d_conv, input_map, kernal):
    d_kernal = np.zeros_like(kernal)
    d_input = np.zeros_like(input_map)
    for i in range(d_conv.shape[0]):   # iterate over d_conv, not d_input
      for j in range(d_conv.shape[1]):
        d_kernal += input_map[i:i+self.kernal_size, j:j+self.kernal_size] * d_conv[i,j]
        d_input[i:i+self.kernal_size, j:j+self.kernal_size] += kernal * d_conv[i,j]
    return d_kernal, d_input
  """

  MAIN TRAING // PUTTING EVERYTHING TOGETHOR

  """
  def fit(self, training_imgs, correct_values, epochs = 30):
    self.input_array = np.pad(training_imgs[0], pad_width=self.pad_width, constant_values=self.constant_value)

    #setting up layer one and two for initally setting FC layer weights and biases
    layer_one, _, _ = self.layer_one()
    layer_two, _, _ = self.layer_two(layer_one)
    flat = self.flatten(layer_two)

    #setting FC layers weights and biases
    self.flattened_size = flat.size
    self.fc_weights = np.random.randn(self.num_classes, self.flattened_size) * 0.01
    self.fc_biases = np.zeros(self.num_classes)

    #training loop
    for epoch in range(epochs):
      total_loss = 0.0
      correct = 0

      #shuffle training data
      indices = np.random.permutation(len(training_imgs))

      for idx in indices:
        img = training_imgs[idx]
        label = correct_values[idx]

        self.input_array = np.pad(img, pad_width=self.pad_width, constant_values=self.constant_value)

        """

        FORWARD PASS

        """
        layer1_out, layer1_pre_relu, layer1_masks = self.layer_one()
        layer2_out, layer2_pre_relu, layer2_masks = self.layer_two(layer1_out)
        flat = self.flatten(layer2_out)

        #FC layer and SOFTMAX
        z = np.dot(self.fc_weights, flat) + self.fc_biases
        exp_z = np.exp(z - np.max(z))
        prediction = exp_z/np.sum(exp_z)

        #cross entropy loss
        loss = -np.sum(label * np.log(prediction +1e-9))
        total_loss += loss
        if np.argmax(prediction) == np.argmax(label):
          correct += 1

        """

        BACKWARD PASS

        """

        #fully connected layer derivatives
        d_fc_out = prediction - label
        d_fc_weights = d_fc_out[:, None] @ flat[None, :]
        d_fc_biases = d_fc_out
        d_flat = self.fc_weights.T @ d_fc_out
        d_l2_out = d_flat.reshape(layer2_out.shape)

        # --- Backprop through layer two ---
        d_l1_out = np.zeros_like(layer1_out)
        d_kernals_l2 = np.zeros_like(self.kernals_layer_two)
        d_biases_l2 = np.zeros_like(self.biases_layer_two)

        for k in range(self.layer_two_size):
          d_activated = self.pool_backward(d_l2_out[k], layer2_masks[k])
          d_conv = self.RELU_backward(d_activated, layer2_pre_relu[k])
          d_biases_l2[k] = np.sum(d_conv)
          for c in range(self.layer_one_size):
            d_k, d_inp = self.conv_backwards(d_conv, layer1_out[c], self.kernals_layer_two[k][c])
            d_kernals_l2[k][c] += d_k
            d_l1_out[c] += d_inp

        # --- Backprop through layer one ---
        d_kernals_l1 = np.zeros_like(self.kernals_layer_one)
        d_biases_l1 = np.zeros_like(self.biases_layer_one)
        for k in range(self.layer_one_size):
          d_activated = self.pool_backward(d_l1_out[k], layer1_masks[k])
          d_conv = self.RELU_backward(d_activated, layer1_pre_relu[k])
          d_biases_l1[k] = np.sum(d_conv)
          d_k, _ = self.conv_backwards(d_conv, self.input_array, self.kernals_layer_one[k][0])
          d_kernals_l1[k][0] += d_k

        # Gradient descent updating
        self.fc_weights    -= self.learning_rate * d_fc_weights
        self.fc_biases     -= self.learning_rate * d_fc_biases
        self.kernals_layer_two -= self.learning_rate * d_kernals_l2
        self.biases_layer_two  -= self.learning_rate * d_biases_l2
        self.kernals_layer_one -= self.learning_rate * d_kernals_l1
        self.biases_layer_one  -= self.learning_rate * d_biases_l1

      acc = correct / len(training_imgs) * 100
      print(f"Epoch {epoch + 1}/{epochs} | Loss: {total_loss / len(training_imgs):.4f} | Accuracy: {acc:.2f}%")

  def predict(self, img):
    """Return class probabilities for a single (unpadded) image."""
    self.input_array = np.pad(img, pad_width=self.pad_width,
                              constant_values=self.constant_value)
    l1_out, _, _ = self.layer_one()
    l2_out, _, _ = self.layer_two(l1_out)
    flat = self.flatten(l2_out)
    z = np.dot(self.fc_weights, flat) + self.fc_biases
    exp_z = np.exp(z - np.max(z))
    return exp_z / np.sum(exp_z)


In [ ]:
x_small = x_train[:500]
y_small = y_train_oh[:500]

model = CNN(x_small[0], kernal_size=3, layer_one_size=4, layer_two_size=8, learning_rate=0.01)
model.fit(x_small, y_small, epochs=10)

# predict a single image
probs = model.predict(x_test[0])
predicted_class = np.argmax(probs)
actual_class = y_test[0]

print(f"Predicted: {predicted_class}, Actual: {actual_class}")

correct = 0
for i in range(100):
    pred = np.argmax(model.predict(x_test[i]))
    if pred == y_test[i]:
        correct += 1

print(f"Test accuracy: {correct / len(x_test) * 100:.2f}%")

Epoch 1/10 | Loss: 1.0728 | Accuracy: 65.00%
Epoch 2/10 | Loss: 0.3894 | Accuracy: 88.00%
Epoch 3/10 | Loss: 0.2562 | Accuracy: 92.60%
Epoch 4/10 | Loss: 0.1972 | Accuracy: 93.40%
Epoch 5/10 | Loss: 0.1612 | Accuracy: 94.20%
Epoch 6/10 | Loss: 0.1275 | Accuracy: 95.80%
Epoch 7/10 | Loss: 0.0802 | Accuracy: 97.20%
Epoch 8/10 | Loss: 0.0712 | Accuracy: 98.00%
Epoch 9/10 | Loss: 0.0243 | Accuracy: 99.60%
Epoch 10/10 | Loss: 0.0384 | Accuracy: 98.60%
Predicted: 7, Actual: 7
Test accuracy: 0.94%


In [ ]:
correct = 0
for i in range(len(x_small)):
    pred = np.argmax(model.predict(x_small[i]))
    if pred == np.argmax(y_small[i]):
        correct += 1
print(f"Train accuracy: {correct / len(x_small) * 100:.2f}%")


Train accuracy: 99.00%
